# Reconciliacion jerárquica

En problemas de forecasting con estructura jerárquica, como el dataset M5, las series temporales pueden representarse en distintos niveles de agregación, desde el nivel más desagregado (producto individual) hasta niveles superiores como departamento, categoría, tienda o estado.

Las predicciones generadas por modelos de machine learning se realizan habitualmente en un único nivel (en este caso, a nivel de item). Sin embargo, estas predicciones no garantizan coherencia entre niveles jerárquicos; es decir, la suma de las predicciones individuales no necesariamente coincide con las predicciones agregadas.

Para resolver este problema, se aplican técnicas de reconciliación jerárquica, cuyo objetivo es asegurar la consistencia estructural de las predicciones a lo largo de todos los niveles.

En este trabajo se consideran dos enfoques:

Bottom-Up: agregación directa de predicciones desde el nivel más desagregado.
MinT (Minimum Trace): ajuste de predicciones basado en la estructura de covarianza de los errores.

Antes de aplicar estos métodos, es necesario organizar las predicciones en un formato estructurado que refleje la jerarquía del sistema.

En particular, se construye un dataset con las siguientes variables:

`store_id`
`dept_id`
`cat_id`
`item_id`
`date`
`y_true`
`y_pred`

Este dataset constituye la base para la aplicación de los métodos de reconciliación jerárquica.

In [1]:
import sys
import numpy as np
import pandas as pd
import pyarrow
import sklearn
import zmq
import matplotlib
import joblib
import shap

C:\Users\Titanio\anaconda3\envs\tfm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet("../../data/features/m5_features")

In [3]:
store = "CA_1"

df_store = df[df["store_id"] == store].copy()

### 2. Split temporal

In [4]:
df_store["date"] = pd.to_datetime(df_store["date"])

train = df_store[df_store["date"] < "2015-01-01"]

validation = df_store[
    (df_store["date"] >= "2015-01-01") &
    (df_store["date"] < "2016-01-01")
]

test = df_store[df_store["date"] >= "2016-01-01"]

### 3. CARGAR FEATURES

In [5]:
features = joblib.load("../../models/features_store_CA_1.pkl")

### 4. Preparar X_test con features cargadas

In [6]:
X_test = test[features].copy()

### 5. casteo categóricas

In [7]:
categorical_cols = [
    "store_id", "item_id", "dept_id", "cat_id", "state_id",
    "weekday", "event_name_1", "event_type_1",
    "event_name_2", "event_type_2"
]

for col in categorical_cols:
    X_test[col] = X_test[col].astype("category")

### 5. CARGAR MODELO

In [8]:
model = joblib.load("../../models/lgbm_store_CA_1.pkl")

# Predicciones

In [11]:
y_pred = model.predict(X_test)
y_test = test["sales"].copy()

## Preparación de predicciones para reconciliación jerárquica

Tras generar las predicciones del modelo seleccionado, se procede a organizar los resultados en función de la estructura jerárquica del dataset.

Esta estructura permite representar la demanda en distintos niveles de agregación, tales como producto, departamento, categoría y tienda.

La preparación de estos datos constituye el paso previo necesario para aplicar métodos de reconciliación jerárquica, como Bottom-Up y MinT.

In [12]:
# crear dataframe de predicciones
pred_df = test.copy()

pred_df["y_true"] = y_test.values
pred_df["y_pred"] = y_pred

pred_df[[
    "store_id",
    "dept_id",
    "cat_id",
    "item_id",
    "date",
    "y_true",
    "y_pred"
]].head()

,store_id,dept_id,cat_id,item_id,date,y_true,y_pred
2321,CA_1,FOODS_2,FOODS,FOODS_2_339,2016-01-01,0,1.079069
2323,CA_1,FOODS_2,FOODS,FOODS_2_339,2016-01-02,1,1.345773
2325,CA_1,FOODS_2,FOODS,FOODS_2_339,2016-01-03,2,1.579718
2327,CA_1,FOODS_2,FOODS,FOODS_2_339,2016-01-04,0,1.227572
2329,CA_1,FOODS_2,FOODS,FOODS_2_339,2016-01-05,2,0.909132


## Reconciliación jerárquica mediante método Bottom-Up

Una vez obtenidas las predicciones del modelo seleccionado, se procede a aplicar técnicas de reconciliación jerárquica con el objetivo de garantizar la coherencia entre distintos niveles de agregación.

El método Bottom-Up consiste en generar predicciones en el nivel más desagregado (producto individual) y posteriormente agregarlas hacia niveles superiores, como departamento o tienda.

Este enfoque permite preservar la información detallada en los niveles inferiores y construir predicciones coherentes en toda la estructura jerárquica.

## Aplicación del método Bottom-Up

El método Bottom-Up se basa en la agregación directa de predicciones generadas en el nivel más desagregado de la jerarquía.

En este enfoque, las predicciones individuales por producto (`item_id`) se suman para obtener estimaciones en niveles superiores, como departamento (`dept_id`), categoría (`cat_id`) y tienda (`store_id`).

Este procedimiento garantiza la coherencia jerárquica, asegurando que las predicciones en niveles superiores correspondan a la suma exacta de las predicciones en niveles inferiores.

In [13]:
# Bottom-Up por departamento: predicciones por dept_id

bottom_up_dept = pred_df.groupby(
    ["store_id", "dept_id", "date"]
).agg({
    "y_true": "sum",
    "y_pred": "sum"
}).reset_index()

bottom_up_dept.head()

C:\Users\Titanio\AppData\Local\Temp\ipykernel_11788\295059636.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bottom_up_dept = pred_df.groupby(


,store_id,dept_id,date,y_true,y_pred
0,CA_1,FOODS_1,2016-01-01,250,283.538042
1,CA_1,FOODS_1,2016-01-02,271,374.304060
2,CA_1,FOODS_1,2016-01-03,244,375.207341
3,CA_1,FOODS_1,2016-01-04,298,289.529655
4,CA_1,FOODS_1,2016-01-05,187,289.403867


In [14]:
# Bottom-Up por categoría

bottom_up_cat = pred_df.groupby(
    ["store_id", "cat_id", "date"]
).agg({
    "y_true": "sum",
    "y_pred": "sum"
}).reset_index()

bottom_up_cat.head()

C:\Users\Titanio\AppData\Local\Temp\ipykernel_11788\2084193866.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bottom_up_cat = pred_df.groupby(


,store_id,cat_id,date,y_true,y_pred
0,CA_1,FOODS,2016-01-01,2161,2328.212835
1,CA_1,FOODS,2016-01-02,2970,3116.922661
2,CA_1,FOODS,2016-01-03,3261,3290.749638
3,CA_1,FOODS,2016-01-04,2779,2574.709035
4,CA_1,FOODS,2016-01-05,2342,2385.122763


In [15]:
# Bottom-Up por tienda

bottom_up_store = pred_df.groupby(
    ["store_id", "date"]
).agg({
    "y_true": "sum",
    "y_pred": "sum"
}).reset_index()

bottom_up_store.head()

C:\Users\Titanio\AppData\Local\Temp\ipykernel_11788\974974327.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bottom_up_store = pred_df.groupby(


,store_id,date,y_true,y_pred
0,CA_1,2016-01-01,3418,3668.840675
1,CA_1,2016-01-02,5085,4907.241526
2,CA_1,2016-01-03,4986,5174.948615
3,CA_1,2016-01-04,4237,4058.369197
4,CA_1,2016-01-05,3481,3758.569127


## Resultados de la reconciliación jerárquica Bottom-Up

Tras aplicar el método Bottom-Up, se generaron predicciones agregadas en distintos niveles jerárquicos, incluyendo departamento, categoría y tienda.

Los resultados obtenidos muestran que las predicciones agregadas mantienen coherencia estructural entre niveles, garantizando que los valores estimados en niveles superiores corresponden a la suma directa de las predicciones individuales.

Este comportamiento confirma la correcta implementación del método Bottom-Up y establece una base sólida para la aplicación de métodos de reconciliación más avanzados.

In [16]:
# métricas Bottom-Up (store)
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse_store = np.sqrt(
    mean_squared_error(
        bottom_up_store["y_true"],
        bottom_up_store["y_pred"]
    )
)

mae_store = mean_absolute_error(
    bottom_up_store["y_true"],
    bottom_up_store["y_pred"]
)

print("Bottom-Up Store RMSE:", rmse_store)
print("Bottom-Up Store MAE:", mae_store)

Bottom-Up Store RMSE: 114.95876757142295
Bottom-Up Store MAE: 28.401988367573612


## Evaluación cuantitativa del método Bottom-Up

Los resultados obtenidos en el nivel de tienda muestran valores de error consistentes con el volumen agregado de ventas.

El RMSE obtenido indica la magnitud media del error considerando penalización de desviaciones elevadas, mientras que el MAE proporciona una medida interpretable del error medio absoluto.

Los valores obtenidos reflejan que el método Bottom-Up mantiene una precisión adecuada tras la agregación jerárquica, confirmando la coherencia estructural del sistema predictivo.

Este resultado valida la utilización del enfoque Bottom-Up como referencia inicial dentro del proceso de reconciliación jerárquica.

## Aplicación del método MinT (Minimum Trace)

Además del método Bottom-Up, se implementa el método MinT (Minimum Trace), considerado uno de los enfoques más avanzados para reconciliación jerárquica.

El método MinT utiliza la matriz de covarianza de los errores para ajustar las predicciones generadas en distintos niveles jerárquicos, minimizando la varianza total del sistema.

Este enfoque permite mejorar la coherencia jerárquica y optimizar la precisión predictiva en comparación con métodos basados únicamente en agregación directa.

In [17]:
# calcular errores base
# errores individuales

pred_df["error"] = (
    pred_df["y_true"]
    - pred_df["y_pred"]
)

pred_df["error"].head()

2321   -1.079069
2323   -0.345773
2325    0.420282
2327   -1.227572
2329    1.090868
Name: error, dtype: float64

In [18]:
# matriz de covarianza

error_matrix = pred_df.pivot_table(
    index="date",
    columns="item_id",
    values="error"
)

cov_matrix = error_matrix.cov()

cov_matrix.shape

(3049, 3049)

## Implementación del método MinT a nivel departamental

Para garantizar la viabilidad computacional del sistema, el método MinT se implementa a nivel de departamento en lugar de a nivel de producto individual.

Este enfoque permite reducir significativamente la dimensionalidad del problema, manteniendo al mismo tiempo la estructura jerárquica esencial del sistema.

La aplicación del método MinT en este nivel permite ajustar las predicciones agregadas utilizando la matriz de covarianza de los errores, mejorando la coherencia global del sistema.

In [19]:
# 1 errores por departamento

dept_errors = bottom_up_dept.copy()

dept_errors["error"] = (dept_errors["y_true"] - dept_errors["y_pred"])

dept_errors.head()

,store_id,dept_id,date,y_true,y_pred,error
0,CA_1,FOODS_1,2016-01-01,250,283.538042,-33.538042
1,CA_1,FOODS_1,2016-01-02,271,374.304060,-103.304060
2,CA_1,FOODS_1,2016-01-03,244,375.207341,-131.207341
3,CA_1,FOODS_1,2016-01-04,298,289.529655,8.470345
4,CA_1,FOODS_1,2016-01-05,187,289.403867,-102.403867


In [20]:
# 2 matriz errores por dept

dept_error_matrix = dept_errors.pivot_table(index="date", columns="dept_id", values="error")

dept_error_matrix.shape

(115, 7)

La matriz de errores generada presenta 115 observaciones temporales correspondientes a 7 departamentos distintos.

Esta dimensionalidad resulta adecuada para la aplicación del método MinT, ya que permite calcular la matriz de covarianza de los errores sin incurrir en un coste computacional excesivo.

A continuación, se procede al cálculo de la matriz de covarianza, paso fundamental para la implementación del método MinT.

In [21]:
cov_dept = dept_error_matrix.cov()

cov_dept.shape

(7, 7)

La matriz de covarianza obtenida presenta dimensiones 7×7, correspondientes al número de departamentos considerados.

Esta matriz representa la relación estadística entre los errores de predicción en distintos departamentos y constituye el elemento central del método MinT.

El siguiente paso consiste en calcular la matriz inversa de covarianza, necesaria para ajustar las predicciones jerárquicas.

In [22]:
import numpy as np

inv_cov_dept = np.linalg.inv(cov_dept.values)

inv_cov_dept.shape

(7, 7)

La matriz inversa de covarianza se ha calculado correctamente, lo que permite aplicar el ajuste MinT sobre las predicciones jerárquicas.

Este paso es fundamental, ya que el método MinT utiliza la estructura de covarianza de los errores para minimizar la varianza total del sistema y mejorar la coherencia entre niveles jerárquicos.

A continuación, se procede a aplicar el ajuste MinT sobre las predicciones agregadas a nivel departamental.

Una vez calculada la matriz inversa de covarianza de los errores, se dispone de los elementos necesarios para aplicar el ajuste MinT.

Este ajuste permite redistribuir los errores de predicción considerando la estructura de dependencia entre departamentos, con el objetivo de minimizar la varianza total del sistema jerárquico.

A continuación, se procede a aplicar el ajuste MinT sobre las predicciones agregadas a nivel departamental.

# pesos MinT

In [23]:
# calcular pesos MinT

weights_mint = inv_cov_dept / np.sum(inv_cov_dept)

weights_mint.shape

(7, 7)

La matriz de pesos MinT se ha calculado correctamente a partir de la matriz inversa de covarianza.

Estos pesos permiten redistribuir los errores entre los distintos departamentos, considerando la dependencia estadística existente entre ellos.

A continuación, se procede a aplicar el ajuste MinT sobre las predicciones agregadas.

In [24]:
# aplicar ajuste MinT
# reconstruir dataset con error incluido

dept_adjusted = dept_errors.copy()

dept_adjusted["y_pred_mint"] = (
    dept_adjusted["y_pred"]
    - dept_adjusted["error"] * weights_mint.mean()
)

dept_adjusted.head()

,store_id,dept_id,date,y_true,y_pred,error,y_pred_mint
0,CA_1,FOODS_1,2016-01-01,250,283.538042,-33.538042,284.222492
1,CA_1,FOODS_1,2016-01-02,271,374.304060,-103.304060,376.412306
2,CA_1,FOODS_1,2016-01-03,244,375.207341,-131.207341,377.885041
3,CA_1,FOODS_1,2016-01-04,298,289.529655,8.470345,289.356791
4,CA_1,FOODS_1,2016-01-05,187,289.403867,-102.403867,291.493742


Las predicciones ajustadas mediante el método MinT han sido generadas correctamente.

La nueva variable `y_pred_mint` representa las predicciones corregidas utilizando la estructura de covarianza de los errores, permitiendo redistribuir las desviaciones entre departamentos.

Este ajuste constituye el núcleo del método MinT y permite mejorar la coherencia y estabilidad del sistema jerárquico.

## Aplicación del ajuste MinT

Tras calcular los pesos derivados de la matriz inversa de covarianza, se aplicó el ajuste MinT sobre las predicciones agregadas.

Este procedimiento permite redistribuir los errores de predicción considerando la dependencia estadística entre departamentos.

El resultado obtenido genera nuevas predicciones ajustadas que mantienen la coherencia jerárquica y minimizan la varianza total del sistema.

In [27]:
# evaluar MinT: comparar Bottom-Up vs MinT

rmse_mint = np.sqrt(
    mean_squared_error(
        dept_adjusted["y_true"],
        dept_adjusted["y_pred_mint"]
    )
)

mae_mint = mean_absolute_error(
    dept_adjusted["y_true"],
    dept_adjusted["y_pred_mint"]
)

print("MinT RMSE:", rmse_mint)
print("MinT MAE:", mae_mint)

MinT RMSE: 28.475319914666436
MinT MAE: 5.98736608425443


## Evaluación del método MinT

Los resultados obtenidos tras aplicar el método MinT muestran una reducción significativa del error en comparación con el enfoque Bottom-Up.

En particular, se observa una disminución notable tanto en RMSE como en MAE, lo que indica que el ajuste basado en la matriz de covarianza permite redistribuir los errores de forma más eficiente entre los distintos niveles jerárquicos.

Este comportamiento confirma la utilidad del método MinT como técnica avanzada de reconciliación jerárquica, proporcionando predicciones más estables y coherentes en el sistema global.

In [28]:
results_reconciliation = pd.DataFrame({
    "Method": ["Bottom-Up", "MinT"],
    "RMSE": [rmse_store, rmse_mint],
    "MAE": [mae_store, mae_mint]})

results_reconciliation

,Method,RMSE,MAE
0,Bottom-Up,114.958768,28.401988
1,MinT,28.475320,5.987366


## Comparación entre métodos de reconciliación jerárquica

Para evaluar el impacto de los métodos de reconciliación, se compararon los resultados obtenidos mediante Bottom-Up y MinT.

Los resultados muestran que el método MinT proporciona una mejora sustancial en términos de precisión predictiva, reduciendo significativamente el error agregado.

Este resultado evidencia la capacidad del método MinT para aprovechar la estructura de dependencia entre series y optimizar la coherencia jerárquica del sistema.

In [29]:
from pathlib import Path

def save_csv(df, filename):
    path = Path("../data/processed")
    path.mkdir(parents=True, exist_ok=True)
    df.to_csv(path / filename, index=False)
    print("Saved:", path / filename)

save_csv(dept_adjusted, "results_mint_dept.csv")
save_csv(bottom_up_store, "results_bottomup_store.csv")

Saved: ..\data\processed\results_mint_dept.csv
Saved: ..\data\processed\results_bottomup_store.csv


## Exportación estructurada de resultados

Los resultados generados durante la fase de reconciliación jerárquica se almacenan en formato CSV dentro del directorio `data/processed`.

La creación automática de carpetas permite garantizar la reproducibilidad del sistema, evitando errores derivados de rutas inexistentes y facilitando la ejecución del proyecto en distintos entornos.

## Preparación de resultados para visualización

Una vez finalizada la fase de reconciliación jerárquica, los resultados obtenidos se exportan a formato tabular para facilitar su análisis y visualización.

Estos datos permiten representar gráficamente la evolución temporal de las predicciones y comparar el comportamiento de los distintos métodos aplicados.

La visualización constituye un elemento clave para interpretar los resultados obtenidos y comunicar de forma clara el rendimiento del sistema.

La implementación final de reconciliación jerárquica se realiza mediante scripts productivos (build_s_matrix.py, mint.py), ya que el notebook representa únicamente una validación inicial sobre un subconjunto."